In [88]:
import re                               # for document class
import xml.etree.ElementTree as ET      # for parsing xml
import os
import pandas as pd
# text processor
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
# end text processor
# encoplot engine
import subprocess
from sentence_transformers import SentenceTransformer, util
import torch
# end encoplot engine
import shutil                             # copy dataset
# for dbscan
from sklearn.cluster import DBSCAN
import numpy as np
# prefilter with tfidf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import math

In [5]:
from google.colab import drive

# mount the drive
drive.mount('/content/drive', force_remount=True)

# define the paths for the dataset
BASE_PATH = "/content/drive/MyDrive/PAN11/external-detection-corpus"

SOURCE_PATH = os.path.join(BASE_PATH, "source-document")
SUSPICIOUS_PATH = os.path.join(BASE_PATH, "suspicious-document")

print(f"Lookin for data in: {BASE_PATH}")

Mounted at /content/drive
Lookin for data in: /content/drive/MyDrive/PAN11/external-detection-corpus


In [6]:
# define the paths for the encoplot code
ENCOPLOT_PATH = "/content/drive/MyDrive/utils/encoplot.c"
EXECUTABLE_PATH = "./encoplot_engine"

SBERT_THRESHOLD = 0.60
CONFIDENCE_THRESHOLD = 0.75
TFIDF_THRESHOLD = 0.05
EPS = 6000
MIN_SAMPLE = 2
MAX_ANCHORS = 300

In [7]:
!gcc -O3 {ENCOPLOT_PATH} -o {EXECUTABLE_PATH}

In [26]:
%%writefile mass_encoplot.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <omp.h>

#define halfword_t unsigned long long
typedef struct { halfword_t lo, hi; } word_t;

#define eqword(x,y) (x.lo==y.lo && x.hi==y.hi)
#define ltword(x,y) (x.hi<y.hi || (x.hi==y.hi && x.lo<y.lo))
#define readat(x,y,z) {x.lo=*(halfword_t *)(y+z); x.hi=*(halfword_t *)(y+z+8);}

typedef struct {
    unsigned char *buf;
    int *ix;
    int len;
} processed_doc;

void simpler_rsort(unsigned char *x, int l, int DEPTH, int *retval) {
    int NN = l - DEPTH + 1;
    if (NN <= 0) return;
    int *ox = (int*)malloc(NN * sizeof(int));
    int counters[256], startpos[256];
    for (int i = 0; i < NN; i++) retval[i] = i;
    for (int j = 0; j < DEPTH; j++) {
        memset(counters, 0, sizeof(counters));
        for (int i = 0; i < NN; i++) counters[x[j + retval[i]]]++;
        int sp = 0;
        for (int k = 0; k < 256; k++) { startpos[k] = sp; sp += counters[k]; }
        for (int i = 0; i < NN; i++) {
            unsigned char c = x[j + retval[i]];
            ox[startpos[c]++] = retval[i];
        }
        memcpy(retval, ox, NN * sizeof(int));
    }
    free(ox);
}

processed_doc process_file(const char *path) {
    processed_doc doc = {NULL, NULL, 0};
    FILE *f = fopen(path, "rb");
    if (!f) return doc;
    fseek(f, 0, SEEK_END);
    doc.len = ftell(f);
    rewind(f);
    doc.buf = (unsigned char *)malloc(doc.len + 20);
    fread(doc.buf, 1, doc.len, f);
    fclose(f);
    doc.ix = (int *)malloc(doc.len * sizeof(int));
    simpler_rsort(doc.buf, doc.len, 16, doc.ix);
    return doc;
}

int main(int argc, char **argv) {
    if (argc < 3) {
        fprintf(stderr, "Utilizare: %s suspicios.txt sursa1.txt sursa2.txt ...\n", argv[0]);
        return 1;
    }

    processed_doc susp = process_file(argv[1]);
    if (!susp.buf) return 1;

    int depth = 16;
    int susp_limit = susp.len - (depth - 1);

    #pragma omp parallel for schedule(dynamic)
    for (int i = 2; i < argc; i++) {
        processed_doc src = process_file(argv[i]);
        if (!src.buf) continue;

        int src_limit = src.len - (depth - 1);
        int i1 = 0, i2 = 0;

        while (i1 < susp_limit && i2 < src_limit) {
            word_t s1; readat(s1, susp.buf, susp.ix[i1]);
            word_t s2; readat(s2, src.buf, src.ix[i2]);

            if (eqword(s1, s2)) {
                #pragma omp critical
                printf("%d %d %d\n", i, susp.ix[i1], src.ix[i2]);
                i1++; i2++;
            } else if (ltword(s1, s2)) { i1++; } else { i2++; }
        }
        free(src.buf); free(src.ix);
    }

    free(susp.buf); free(susp.ix);
    return 0;
}

Overwriting mass_encoplot.c


In [27]:
!gcc -O3 -fopenmp mass_encoplot.c -o mass_encoplot

mass_encoplot.c: In function ‘process_file’:
mass_encoplot.c:47:5: warning: ignoring return value of ‘fread’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   47 |     fread(doc.buf, 1, doc.len, f);
      |     ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [9]:
class Document:
  def __init__(self, doc_id, is_source=True):
    self.numeric_id = int(doc_id)
    self.is_source = is_source

    formatted_id = f"{self.numeric_id:05d}"
    prefix = "source-document" if is_source else "suspicious-document"

    part_number = ((self.numeric_id - 1) // 500) + 1
    part_folder = f"part{part_number}"
    base_folder = SOURCE_PATH if is_source else SUSPICIOUS_PATH

    self.doc_name = f"{prefix}{formatted_id}.txt"
    self.xml_name = f"{prefix}{formatted_id}.xml"

    self.file_path = os.path.join(base_folder, part_folder, self.doc_name)
    self.xml_path = os.path.join(base_folder, part_folder, self.xml_name)

    with open(self.file_path, "r", encoding='utf-8', errors='ignore') as f:
      self.text = f.read()

    self.metadata = self._parse_xml()
    self.language = self.metadata.get('lang', 'english')
    self.segments = []

  def _parse_xml(self):
    meta = {}
    self.plagiarism_features = []
    try:
        tree = ET.parse(self.xml_path)
        root = tree.getroot()

        for feature in root.findall('feature'):
            if feature.get('name') == 'about':
                meta['lang'] = feature.get('lang', 'en')

            if feature.get('name') == 'md5Hash':
                meta['md5'] = feature.get('value')

            if feature.get('name') == 'plagiarism':
                self.plagiarism_features.append(PlagiarismFeature(
                    this_offset=int(feature.get('this_offset')),
                    this_length=int(feature.get('this_length')),
                    source_reference=feature.get('source_reference'),
                    source_offset=int(feature.get('source_offset')),
                    source_length=int(feature.get('source_length')),
                    obfuscation=feature.get('obfuscation', 'none')
                ))
    except Exception as e:
        print(f"Error parsing XML: {e}")
    return meta

In [10]:
class PlagiarismFeature:
    def __init__(self, this_offset, this_length, source_reference,
                 source_offset, source_length, obfuscation):
        self.this_offset = this_offset
        self.this_length = this_length
        self.source_reference = source_reference
        self.source_offset = source_offset
        self.source_length = source_length
        self.obfuscation = obfuscation

    def get_source_id(self):
        return int(self.source_reference
                       .replace("source-document", "")
                       .replace(".txt", ""))

    def __repr__(self):
        return (f"PlagiarismFeature("
                f"offset={self.this_offset}, "
                f"length={self.this_length}, "
                f"source={self.source_reference}, "
                f"obfuscation={self.obfuscation})")

In [11]:
class PredictedSegment:
    def __init__(self, susp_id, src_id, susp_off, susp_len, src_off, src_len, score):
        self.susp_id = susp_id
        self.src_id = src_id
        self.susp_off = susp_off
        self.susp_len = susp_len
        self.src_off = src_off
        self.src_len = src_len
        self.score = score

    def __repr__(self):
        return f"<Match Susp:{self.susp_id} Src:{self.src_id} Score:{self.score:.2f}>"

In [86]:
class EncoplotEngine:
    def __init__(self, executable_path="./mass_encoplot", max_anchors=300):
        self.executable = executable_path
        self.max_anchors = max_anchors

    def run_mass_scan(self, susp_path, source_paths):
        cmd = [self.executable, susp_path] + source_paths
        result = subprocess.run(cmd, capture_output=True, text=True)
        anchors_by_source = {path: [] for path in source_paths}

        if result.returncode != 0:
            print(f"  [!] Encoplot execution error: {result.stderr}")
            return anchors_by_source

        for line in result.stdout.strip().split('\n'):
            if not line: continue
            parts = list(map(int, line.split()))
            if len(parts) == 3:
                s_idx, susp_off, src_off = parts
                anchors_by_source[source_paths[s_idx - 2]].append([susp_off, src_off])
        return anchors_by_source

    def apply_bucketing(self, anchors, doc_len):
        if len(anchors) <= self.max_anchors: return anchors
        bucket_size = max(1, doc_len // self.max_anchors)
        buckets = {}
        for p_susp, p_src in anchors:
            bucket_idx = p_susp // bucket_size
            if bucket_idx not in buckets: buckets[bucket_idx] = (p_susp, p_src)
        return list(buckets.values())

def cluster_results_dbscan(hits, eps=5000, min_samples=2, confidence_threshold=0.75):
    if not hits: return []
    X = np.array([[h.susp_off, h.src_off] for h in hits])
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit(X).labels_

    merged = []
    for label in set(labels):
        if label == -1: continue
        c_hits = [hits[i] for i in range(len(hits)) if labels[i] == label]
        merged.append(PredictedSegment(
            c_hits[0].susp_id, c_hits[0].src_id,
            min(h.susp_off for h in c_hits), max(h.susp_off + h.susp_len for h in c_hits) - min(h.susp_off for h in c_hits),
            min(h.src_off for h in c_hits), max(h.src_off + h.src_len for h in c_hits) - min(h.src_off for h in c_hits),
            sum(h.score for h in c_hits) / len(c_hits)
        ))
    for i, label in enumerate(labels):
        if label == -1 and hits[i].score >= confidence_threshold:
            merged.append(hits[i])
    return merged

In [13]:
class SemanticAnalyzer:
  def __init__(self, model_name='paraphrase-multilingual-mpnet-base-v2'):  # paraphrase-multilingual-MiniLM-L12-v2
    self.device = "cuda" if torch.cuda.is_available() else "cpu"
    self.model = SentenceTransformer(model_name).to(self.device)

In [14]:
def get_safe_fragment(text, offset, win=450):
        if not text:
            return "", 0, 0

        text_len = len(text)
        start = max(0, min(offset, text_len - 1))

        while start > 0:
            if text[start-1] in [' ', '\n', '\t']:
                break
            start -= 1

        end = min(text_len, start + win)

        while end < len(text) and text[end] not in [' ', '\n', '.', '!', '?']:
            end += 1

        fragment = text[start:end]
        return fragment, start, end - start

In [87]:
class ValidationMetrics:
    def __init__(self):
        pass

    def compute_granularity(self, gt_fragments, detected_fragments):
        if not gt_fragments:
            return 1.0
        total_gran = 0
        for gt_start, gt_end in gt_fragments:
            overlapping = 0
            for det_start, det_end in detected_fragments:
                if det_start < gt_end and det_end > gt_start:
                    overlapping += 1
            total_gran += max(1, overlapping)
        return total_gran / len(gt_fragments)

    def calculate_metrics(self, ground_truth_features, predicted_segments):
        metrics = {
            "recall": 0.0,
            "precision": 0.0,
            "f1": 0.0,
            "granularity": 1.0,
            "plagdet": 0.0
        }

        if not ground_truth_features:
            metrics["recall"] = 1.0
            return metrics

        # recall
        total_recall = 0
        for gt in ground_truth_features:
            covered_len = 0
            for pred in predicted_segments:
                intersect_start = max(gt.this_offset, pred.susp_off)
                intersect_end = min(gt.this_offset + gt.this_length, pred.susp_off + pred.susp_len)
                if intersect_end > intersect_start:
                    covered_len += (intersect_end - intersect_start)
            total_recall += (covered_len / gt.this_length)
        metrics["recall"] = total_recall / len(ground_truth_features)

        # precision
        if predicted_segments:
            total_precision = 0
            for pred in predicted_segments:
                covered_len = 0
                for gt in ground_truth_features:
                    intersect_start = max(gt.this_offset, pred.susp_off)
                    intersect_end = min(gt.this_offset + gt.this_length, pred.susp_off + pred.susp_len)
                    if intersect_end > intersect_start:
                        covered_len += (intersect_end - intersect_start)
                total_precision += (covered_len / pred.susp_len)
            metrics["precision"] = total_precision / len(predicted_segments)

        # granularity
        gt_tuples = [(f.this_offset, f.this_offset + f.this_length) for f in ground_truth_features]
        det_tuples = [(d.susp_off, d.susp_off + d.susp_len) for d in predicted_segments]
        metrics["granularity"] = self.compute_granularity(gt_tuples, det_tuples)

        # f1 and plagDet
        if metrics["precision"] + metrics["recall"] > 0:
            metrics["f1"] = 2 * (metrics["precision"] * metrics["recall"]) / (metrics["precision"] + metrics["recall"])

        metrics["plagdet"] = metrics["f1"] / math.log2(1 + metrics["granularity"]) if metrics["granularity"] > 0 else 0

        return {k: round(v, 4) for k, v in metrics.items()}


    def calculate_global_score(self, all_results_list):
      relevant_results = [res for res in all_results_list if not (res['recall'] == 1.0 and res['precision'] == 0.0)]

      if not relevant_results:
        print("\n" + "!"*60)
        print("  NO RELEVANT DATA TO CALCULATE GLOBAL SCORE")
        print("  (All documents were original and correctly identified as such)")
        print("!"*60)
        return None

      n = len(relevant_results)
      avg_metrics = {k: sum(res[k] for res in relevant_results) / n for k in relevant_results[0].keys()}

      print("\n" + "="*60)
      print(f"GLOBAL SCORE (Macro-average on {n} relevant documents)")
      print("-" * 60)
      print(f"  Precision    : {avg_metrics['precision']:.4f}")
      print(f"  Recall       : {avg_metrics['recall']:.4f}")
      print(f"  F1-Score     : {avg_metrics['f1']:.4f}")
      print(f"  Granularity  : {avg_metrics['granularity']:.4f}")
      print(f"  PlagDet      : {avg_metrics['plagdet']:.4f}")
      print("="*60 + "\n")

      return avg_metrics


In [16]:
def get_path(doc_id, is_source=True):
    prefix = "source" if is_source else "suspicious"
    part_num = ((int(doc_id) - 1) // 500) + 1
    file_name = f"{prefix}-document{int(doc_id):05d}.txt"
    return os.path.join(BASE_PATH, f"{prefix}-document", f"part{part_num}", file_name)

In [17]:
def extract_sample(n_suspicious):
    suspicious_docs = []
    source_ids_needed = set()

    for i in range(1, n_suspicious + 1):
        try:
            doc = Document(i, is_source=False)
            suspicious_docs.append(doc)

            for pf in doc.plagiarism_features:
                src_id = int(pf.source_reference
                               .replace("source-document", "")
                               .replace(".txt", ""))
                source_ids_needed.add(src_id)

        except Exception as e:
            print(f"[!] Suspicious {i:05d} error: {e}")
            continue

    print(f"Loaded suspicious docs : {len(suspicious_docs)}")
    print(f"Unique sources : {len(source_ids_needed)}")

    return suspicious_docs, sorted(source_ids_needed)

In [18]:
encoplot = EncoplotEngine()
analyzer = SemanticAnalyzer()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [19]:
def prefilter_with_tfidf(texts_susp, texts_src, all_meta, tfidf_threshold=0.05):
    print(f"Prefilltering with tf-idf on {len(texts_susp)} pairs")

    all_texts = texts_susp + texts_src
    vectorizer = TfidfVectorizer(max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(all_texts)

    n = len(texts_susp)
    susp_matrix = tfidf_matrix[:n]
    src_matrix  = tfidf_matrix[n:]

    filtered_susp, filtered_src, filtered_meta = [], [], []

    BATCH = 10000
    for i in range(0, n, BATCH):
        batch_susp = susp_matrix[i:i+BATCH]
        batch_src  = src_matrix[i:i+BATCH]

        scores = np.array(batch_susp.multiply(batch_src).sum(axis=1)).flatten()

        for j, score in enumerate(scores):
            if score >= tfidf_threshold:
                filtered_susp.append(texts_susp[i+j])
                filtered_src.append(texts_src[i+j])
                filtered_meta.append(all_meta[i+j])

    print(f"  after tf-idf: {len(filtered_susp)} remaining pairs "
          f"(crossed out: {n - len(filtered_susp)})")
    return filtered_susp, filtered_src, filtered_meta

In [22]:
def debug_plagiarism_flow(doc_susp, detections, all_meta_pre_tfidf, all_meta_post_tfidf, scores_sbert=None, threshold=0.60):
    print(f"\n--- DETAILED ANALYSIS: {doc_susp.doc_name} ---")

    my_detections = detections.get(doc_susp.numeric_id, [])

    if doc_susp.plagiarism_features:
        print(f"Ground Truth properties: {len(doc_susp.plagiarism_features)} plagiarized fragments.")
        for pf in doc_susp.plagiarism_features:
            gt_start = pf.this_offset
            gt_end = pf.this_offset + pf.this_length
            src_target = pf.get_source_id()

            enc_hits = [m for m in all_meta_pre_tfidf if m['susp_id'] == doc_susp.numeric_id
                        and m['src_id'] == src_target
                        and (m['s_off'] >= gt_start - 200 and m['s_off'] <= gt_end + 200)]
            tfidf_hits = [m for m in all_meta_post_tfidf if m['susp_id'] == doc_susp.numeric_id
                          and m['src_id'] == src_target
                          and (m['s_off'] >= gt_start - 200 and m['s_off'] <= gt_end + 200)]

            best_score = 0.0
            if scores_sbert is not None:
                relevant_indices = [i for i, m in enumerate(all_meta_post_tfidf)
                                   if m['susp_id'] == doc_susp.numeric_id
                                   and m['src_id'] == src_target
                                   and (m['s_off'] >= gt_start - 200 and m['s_off'] <= gt_end + 200)]
                if relevant_indices:
                    best_score = max([scores_sbert[i].item() for i in relevant_indices])

            final_det = [d for d in my_detections if d.src_id == src_target
                         and (d.susp_off >= gt_start - 500 and d.susp_off <= gt_end + 500)]

            status = "✅ DETECTED" if final_det else "❌ MISSED"
            print(f"\nGT Fragment [{gt_start}:{gt_end}] (Source {src_target:05d}) -> {status}")
            print(f"   - Raw anchors (Encoplot): {len(enc_hits)}")
            print(f"   - After TF-IDF filter:    {len(tfidf_hits)}")
            if len(tfidf_hits) > 0:
                print(f"   - Best SBERT score:       {best_score:.4f}")

            if not final_det:
                if not enc_hits:
                    print("   [CAUSE]: Lexical Failure - Encoplot found no anchors.")
                elif not tfidf_hits:
                    print("   [CAUSE]: Statistical Failure - TF-IDF removed the fragment.")
                else:
                    print(f"   [CAUSE]: Semantic Failure - Score {best_score:.4f} < {threshold} or DBSCAN removed it.")
    else:
        print("Original document (according to Ground Truth).")

    for det in my_detections:
        is_real = False
        for pf in doc_susp.plagiarism_features:
            if not (det.susp_off + det.susp_len < pf.this_offset or pf.this_offset + pf.this_length < det.susp_off):
                is_real = True
                break

        if not is_real:
            print(f"\n FALSE POSITIVE DETECTED (Erroneously reported as plagiarism):")
            print(f"   - Position: [{det.susp_off}:{det.susp_off + det.susp_len}] | Source: {det.src_id:05d}")
            print(f"   - SBERT score of the error: {det.score:.4f}")


In [91]:
def run_pipeline(n_suspicious, analyzer, eps=5000, threshold=0.60):
    print("-" * 80)
    print(f"[STEP 1] Extracting sample: {n_suspicious} suspicious files")
    print("-" * 80)
    suspicious_docs, source_ids = extract_sample(n_suspicious)
    print(f"Necessary sources: {len(source_ids)}\n")

    print("[STEP 2] Loading source docs...")
    source_docs = []
    for s_id in source_ids:
        try:
            source_docs.append(Document(s_id, is_source=True))
        except Exception as e:
            print(f"  [!] Error loading source {s_id}: {e}")
    print(f"Sources loaded: {len(source_docs)}\n")

    print("[STEP 3] Encoplot - lexical radar...")
    t3 = time.time()
    engine = EncoplotEngine(max_anchors=300)

    all_texts_susp = []
    all_texts_src = []
    all_meta_pre_tfidf = []

    for doc_susp in suspicious_docs:
        src_paths = [d.file_path for d in source_docs]
        raw_results = engine.run_mass_scan(doc_susp.file_path, src_paths)

        for s_path, anchors in raw_results.items():
            if not anchors:
                continue

            anchors = engine.apply_bucketing(anchors, len(doc_susp.text))

            s_id = int(re.search(r'document(\d+)', s_path).group(1))

            for p_susp, p_src in anchors:
                f_susp, s_off, s_len = get_safe_fragment(doc_susp.text, p_susp)
                f_src, src_off, src_len = get_safe_fragment(doc_susp.text, p_src)

                doc_src = next((d for d in source_docs if d.numeric_id == s_id), None)
                if not doc_src: continue
                f_src, src_off, src_len = get_safe_fragment(doc_src.text, p_src)

                if not f_susp or not f_src:
                    continue

                t_susp = re.sub(r'\s+', ' ', f_susp).strip().lower()
                t_src  = re.sub(r'\s+', ' ', f_src).strip().lower()

                if len(t_susp) > 30:
                    all_texts_susp.append(t_susp)
                    all_texts_src.append(t_src)
                    all_meta_pre_tfidf.append({
                        'susp_id': doc_susp.numeric_id,
                        'src_id':  s_id,
                        's_off':   s_off,
                        's_len':   s_len,
                        'src_off': src_off,
                        'src_len': src_len,
                    })

    t3_elapsed = time.time() - t3
    print(f"Candidate fragments: {len(all_texts_susp)} | Time: {t3_elapsed/60:.1f}min\n")

    filtered_susp, filtered_src, all_meta_post_tfidf = prefilter_with_tfidf(
        all_texts_susp, all_texts_src, all_meta_pre_tfidf, tfidf_threshold=0.05
    )

    print(f"[STEP 5] SBERT batch encoding for {len(filtered_susp)} pairs...")
    t5 = time.time()
    emb_susp = analyzer.model.encode(filtered_susp, convert_to_tensor=True, show_progress_bar=True, batch_size=512)
    emb_src  = analyzer.model.encode(filtered_src,  convert_to_tensor=True, show_progress_bar=True, batch_size=512)
    scores = torch.nn.functional.cosine_similarity(emb_susp, emb_src)
    print(f"Encoding done in {time.time() - t5:.1f}s\n")

    print("[STEP 6] Filtering and clustering with DBSCAN...")
    hits_per_pair = defaultdict(list)
    for i, score in enumerate(scores):
        if score >= threshold:
            m = all_meta_post_tfidf[i]
            hits_per_pair[(m['susp_id'], m['src_id'])].append(
                PredictedSegment(m['susp_id'], m['src_id'], m['s_off'], m['s_len'], m['src_off'], m['src_len'], score.item())
            )

    detections = defaultdict(list)
    for (susp_id, src_id), hits in hits_per_pair.items():
        clustered = cluster_results_dbscan(hits, eps=eps, min_samples=2, confidence_threshold=0.75)
        detections[susp_id].extend(clustered)
    print(f"Pairs with detected plagiarism: {len(hits_per_pair)}\n")

    print("[STEP 7] Generating XML output...")
    os.makedirs("/content/output", exist_ok=True)
    for doc_susp in suspicious_docs:
        root = ET.Element("document", reference=doc_susp.doc_name)
        for det in detections.get(doc_susp.numeric_id, []):
            ET.SubElement(root, "feature",
                name="detected-plagiarism",
                this_offset=str(det.susp_off),
                this_length=str(det.susp_len),
                source_reference=f"source-document{det.src_id:05d}.txt",
                source_offset=str(det.src_off),
                source_length=str(det.src_len)
            )
        ET.ElementTree(root).write(f"/content/output/{doc_susp.doc_name.replace('.txt', '.xml')}", encoding='utf-8', xml_declaration=True)


    validator = ValidationMetrics()
    all_results = []
    print("[STEP 8] Detected vs ground truth...")
    print("-"*60)

    for doc_susp in suspicious_docs:
        m = validator.calculate_metrics(doc_susp.plagiarism_features, detections.get(doc_susp.numeric_id, []))
        all_results.append(m)

        if len(doc_susp.plagiarism_features) > 0 or len(detections.get(doc_susp.numeric_id, [])) > 0:
            print(f"  Suspicious {doc_susp.numeric_id:05d} | GT: {len(doc_susp.plagiarism_features)} | "
                  f"Det: {len(detections.get(doc_susp.numeric_id, []))} | P: {m['precision']:.2f} R: {m['recall']:.2f}")

    validator.calculate_global_score(all_results)

    print("\n" + "="*40)
    print("DETAILED TRACING OF GROUND TRUTH SEGMENTS")
    print("="*40)
    for doc_susp in suspicious_docs:
        debug_plagiarism_flow(doc_susp, detections, all_meta_pre_tfidf, all_meta_post_tfidf, scores_sbert=scores, threshold=threshold)

    return suspicious_docs, source_docs, detections

In [90]:
if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections = run_pipeline(
    n_suspicious=5,
    analyzer=analyzer,
    threshold=0.60
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 5 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 5
Unique sources : 1
Necessary sources: 1

[STEP 2] Loading source docs...
Sources loaded: 1

[STEP 3] Encoplot - lexical radar...
Candidate fragments: 63 | Time: 0.0min

Prefilltering with tf-idf on 63 pairs
  after tf-idf: 62 remaining pairs (crossed out: 1)
[STEP 5] SBERT batch encoding for 62 pairs...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding done in 0.5s

[STEP 6] Filtering and clustering with DBSCAN...
Pairs with detected plagiarism: 1

[STEP 7] Generating XML output...
[STEP 8] Detected vs ground truth...
------------------------------------------------------------
  Suspicious 00005 | GT: 1 | Det: 1 | P: 0.85 R: 0.99

GLOBAL SCORE (Macro-average on 1 relevant documents)
------------------------------------------------------------
  Precision    : 0.8491
  Recall       : 0.9865
  F1-Score     : 0.9127
  Granularity  : 1.0000
  PlagDet      : 0.9127


DETAILED TRACING OF GROUND TRUTH SEGMENTS

--- DETAILED ANALYSIS: suspicious-document00001.txt ---
Original document (according to Ground Truth).

--- DETAILED ANALYSIS: suspicious-document00002.txt ---
Original document (according to Ground Truth).

--- DETAILED ANALYSIS: suspicious-document00003.txt ---
Original document (according to Ground Truth).

--- DETAILED ANALYSIS: suspicious-document00004.txt ---
Original document (according to Ground Truth).

--- DETAILE

In [83]:
import time
from collections import defaultdict

if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections = run_pipeline(
    n_suspicious=10,
    analyzer=analyzer,
    threshold=0.65
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 10 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 10
Unique sources : 3
Necessary sources: 3

[STEP 2] Loading source docs...
Sources loaded: 3

[STEP 3] Encoplot - lexical radar...
Candidate fragments: 3374 | Time: 0.0min

Prefilltering with tf-idf on 3374 pairs
  after tf-idf: 3095 remaining pairs (crossed out: 279)
[STEP 5] SBERT batch encoding for 3095 pairs...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Encoding done in 42.5s

[STEP 6] Filtering and clustering with DBSCAN...
Pairs with detected plagiarism: 7

[STEP 7] Generating XML output...
[STEP 8] Detected vs ground truth...
------------------------------------------------------------
  Suspicious 00005 | GT: 1 | Det: 1 | P: 0.90 R: 0.99
  Suspicious 00007 | GT: 6 | Det: 5 | P: 0.99 R: 0.76
  Suspicious 00010 | GT: 6 | Det: 4 | P: 0.90 R: 0.51

GLOBAL SCORE (Macro-average on 3 relevant documents)
------------------------------------------------------------
  Precision    : 0.9274
  Recall       : 0.7550
  F1-Score     : 0.8184
  Granularity  : 1.1111
  PlagDet      : 0.7661


DETAILED TRACING OF GROUND TRUTH SEGMENTS

--- DETAILED ANALYSIS: suspicious-document00001.txt ---
Original document (according to Ground Truth).

--- DETAILED ANALYSIS: suspicious-document00002.txt ---
Original document (according to Ground Truth).

--- DETAILED ANALYSIS: suspicious-document00003.txt ---
Original document (according to Ground Truth).

--- DE

In [53]:
if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections = run_pipeline(
    n_suspicious=20,
    analyzer=analyzer,
    threshold=0.60
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 20 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 20
Unique sources : 15
Necessary sources: 15

[STEP 2] Loading source docs...
Sources loaded: 15

[STEP 3] Encoplot - lexical radar...
Candidate fragments: 28011 | Time: 0.2min

Prefilltering with tf-idf on 28011 pairs
  after tf-idf: 24987 remaining pairs (crossed out: 3024)
[STEP 5] SBERT batch encoding for 24987 pairs...


Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Encoding done in 333.1s

[STEP 6] Filtering and clustering with DBSCAN...
Pairs with detected plagiarism: 75

[STEP 7] Generating XML output...
[STEP 8] Detected vs ground truth...
------------------------------------------------------------
  Suspicious 00001 | GT: 0 | Detected: 1 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00002 | GT: 0 | Detected: 5 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00004 | GT: 0 | Detected: 3 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00005 | GT: 1 | Detected: 10 | P: 0.17 R: 0.99 F1: 0.29 | Gran: 1.00 PlagDet: 0.2893
  Suspicious 00006 | GT: 0 | Detected: 1 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00007 | GT: 6 | Detected: 5 | P: 0.98 R: 0.91 F1: 0.95 | Gran: 1.33 PlagDet: 0.7743
  Suspicious 00008 | GT: 0 | Detected: 9 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00009 | GT: 0 | Detected: 1 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.

In [54]:
if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections = run_pipeline(
    n_suspicious=30,
    analyzer=analyzer,
    threshold=0.60
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 30 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 30
Unique sources : 21
Necessary sources: 21

[STEP 2] Loading source docs...
Sources loaded: 21

[STEP 3] Encoplot - lexical radar...
Candidate fragments: 51931 | Time: 0.3min

Prefilltering with tf-idf on 51931 pairs
  after tf-idf: 45868 remaining pairs (crossed out: 6063)
[STEP 5] SBERT batch encoding for 45868 pairs...


Batches:   0%|          | 0/90 [00:00<?, ?it/s]

Batches:   0%|          | 0/90 [00:00<?, ?it/s]

Encoding done in 612.8s

[STEP 6] Filtering and clustering with DBSCAN...
Pairs with detected plagiarism: 124

[STEP 7] Generating XML output...
[STEP 8] Detected vs ground truth...
------------------------------------------------------------
  Suspicious 00001 | GT: 0 | Detected: 1 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00002 | GT: 0 | Detected: 5 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00004 | GT: 0 | Detected: 4 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00005 | GT: 1 | Detected: 13 | P: 0.17 R: 0.99 F1: 0.28 | Gran: 1.00 PlagDet: 0.2836
  Suspicious 00006 | GT: 0 | Detected: 2 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00007 | GT: 6 | Detected: 5 | P: 0.98 R: 0.91 F1: 0.95 | Gran: 1.33 PlagDet: 0.7743
  Suspicious 00008 | GT: 0 | Detected: 11 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00009 | GT: 0 | Detected: 1 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 

In [37]:
if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections = run_pipeline(
    n_suspicious=50,
    analyzer=analyzer,
    threshold=0.60
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 50 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 50
Unique sources : 51
Necessary sources: 51

[STEP 2] Loading source docs...
Sources loaded: 51

[STEP 3] Encoplot - lexical radar...
Candidate fragments: 201644 | Time: 1.1min

[STEP 5] SBERT batch encoding for 201644 pairs...


Batches:   0%|          | 0/197 [00:00<?, ?it/s]

Batches:   0%|          | 0/197 [00:00<?, ?it/s]

Encoding done in 2519.4s

[STEP 6] Filtering and clustering with DBSCAN...
Pairs with detected plagiarism: 514

[STEP 7] Generating XML output...
[STEP 8] Detected vs ground truth...
------------------------------------------------------------
  Suspicious 00001 | GT: 0 | Detected: 4 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00002 | GT: 0 | Detected: 13 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00004 | GT: 0 | Detected: 6 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00005 | GT: 1 | Detected: 20 | P: 0.11 R: 0.99 F1: 0.20 | Gran: 1.00 PlagDet: 0.2048
  Suspicious 00006 | GT: 0 | Detected: 5 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00007 | GT: 6 | Detected: 5 | P: 0.98 R: 0.91 F1: 0.95 | Gran: 1.33 PlagDet: 0.7743
  Suspicious 00008 | GT: 0 | Detected: 15 | P: 0.00 R: 0.00 F1: 0.00 | Gran: 1.00 PlagDet: 0.0000
  Suspicious 00009 | GT: 0 | Detected: 4 | P: 0.00 R: 0.00 F1: 0.00 | Gran

In [ ]:
import nbformat

with open('/content/drive/MyDrive/Colab Notebooks/licenta.ipynb', 'r') as f:
    nb = nbformat.read(f, as_version=4)

if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

for cell in nb.cells:
    if 'metadata' in cell:
        if 'executionInfo' in cell.metadata:
            del cell.metadata['executionInfo']

with open('/content/drive/MyDrive/Colab Notebooks/licenta_clean.ipynb', 'w') as f:
    nbformat.write(nb, f)

print("Done!")

Done!
